# Single-Qubit Gates & Superposition — Live Demo Notebook

**Companion notebook to the talk by Fatima Bint-e-Munir**
Qiskit Fall Fest 2026 · University of Engineering & Technology Lahore, Narowal Campus

This notebook follows the slides exactly, in order, so you can run it live during the talk or work
through it afterwards on your own:

- **Part 1** — What exactly is a quantum gate?
- **Part 2** — The Pauli gates: X, Y, Z
- **Part 3** — The Hadamard gate & superposition
- **Part 4** — The phase family: S and T
- **Part 5** — Putting the gates to work

Every gate gets the same four-step treatment from the slides — **Intro & Matrix → Bloch View →
Qiskit Code → Try It** — and every "Predict Before You Peek 🤔" task from the slides has a matching
cell here so you can test your prediction immediately.

💡 **Tip:** Run cells top to bottom. Re-run the whole notebook (`Runtime → Run all` in Colab,
or `Kernel → Restart & Run All` in Jupyter) any time you want a clean slate.

⚙️ **Qiskit version:** this notebook targets Qiskit 2.0. Nothing


In [1]:
!pip install "qiskit>=2.0" pylatexenc -q

ERROR: Ignored the following yanked versions: 1.2.3
ERROR: Ignored the following versions that require a different python version: 1.3.0 Requires-Python >=3.9; 1.3.0b1 Requires-Python >=3.9; 1.3.0rc1 Requires-Python >=3.9; 1.3.0rc2 Requires-Python >=3.9; 1.3.1 Requires-Python >=3.9; 1.3.2 Requires-Python >=3.9; 1.3.3 Requires-Python >=3.9; 1.4.0 Requires-Python >=3.9; 1.4.1 Requires-Python >=3.9; 1.4.2 Requires-Python >=3.9; 1.4.3 Requires-Python >=3.9; 1.4.4 Requires-Python >=3.9; 1.4.5 Requires-Python >=3.9; 1.4.6 Requires-Python >=3.9; 2.0.0 Requires-Python >=3.9; 2.0.0rc1 Requires-Python >=3.9; 2.0.0rc2 Requires-Python >=3.9; 2.0.1 Requires-Python >=3.9; 2.0.2 Requires-Python >=3.9; 2.0.3 Requires-Python >=3.9; 2.1.0 Requires-Python >=3.9; 2.1.0rc1 Requires-Python >=3.9; 2.1.1 Requires-Python >=3.9; 2.1.2 Requires-Python >=3.9; 2.2.0 Requires-Python >=3.9; 2.2.0b1 Requires-Python >=3.9; 2.2.0rc1 Requires-Python >=3.9; 2.2.1 Requires-Python >=3.9; 2.2.2 Requires-Python >=3.9; 2.2.3 

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import numpy as np
import qiskit

%matplotlib inline
np.set_printoptions(precision=3, suppress=True)

print("Qiskit version:", qiskit.__version__)   # should read 2.x


## Part 1 — What Exactly Is a Quantum Gate?

A gate `U` transforms a statevector: `|psi'> = U|psi>` — the same matrix–vector multiplication as
`Operator()` above, just applied to a quantum state. The one extra rule that makes it a *quantum*
gate is **unitarity**: `U† U = I`. This is what guarantees every gate is reversible and preserves
total probability (vector length 1).

Let's check that all six gates from today — X, Y, Z, H, S, T — really are unitary.


In [ ]:
# Build a one-gate circuit for each of the six gates and check U†U = I
gate_names = ["x", "y", "z", "h", "s", "t"]

for name in gate_names:
    qc = QuantumCircuit(1)
    getattr(qc, name)(0)          # e.g. qc.x(0), qc.h(0), ...
    U = Operator(qc).data
    is_unitary = np.allclose(U.conj().T @ U, np.eye(2))
    print(f"{name.upper():>2}:  U†U = I ?  {is_unitary}")

### The Recipe We'll Repeat Six Times

Every gate — X, Y, Z, H, S, T — gets the same four steps below. Once you know the pattern, you can
decode any new gate yourself:

1. **Intro & Matrix** — name, symbol, and the 2×2 matrix that defines it
2. **Bloch View** — which axis it rotates around, and by how much
3. **Qiskit Code** — the one-line call, and the circuit diagram it draws
4. **Try It** — a worked example: what happens to a real state


## Part 2 — The Pauli Gates: X, Y, Z

Named after physicist Wolfgang Pauli, who introduced them in 1927 (decades before quantum
computers!) to describe electron spin.


### X — The Bit-Flip Gate

Flips `|0>` ↔ `|1>`. A π (180°) rotation around the X-axis of the Bloch sphere.

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$

Nickname: the "quantum NOT gate" — the direct analogue of the classical bit-flip, works
on superpositions too.


In [ ]:
X = np.array([[0, 1],
              [1, 0]])

# 1 & 2. Intro, matrix, and Bloch view -----------------------------------
qc = QuantumCircuit(1)
qc.x(0)              # apply X to qubit 0
display(qc.draw(output="mpl"))

sv = Statevector(qc)
print(sv)
plot_bloch_multivector(sv)

🤔 **Predict Before You Peek** — We know X flips `|0>` and `|1>`. So what do you think X does
to `|+> = (|0> + |1>)/√2`, the equal-superposition state? Does it change at all?

Think about it first — then run the cell below to check. We'll build the full intuition for this in
**Part 5**.


In [ ]:
# Build |+> with H, then apply X, and see what comes out
qc = QuantumCircuit(1)
qc.h(0)   # |0> -> |+>
qc.x(0)   # now apply X
print("X|+> =", Statevector(qc))

### Z — The Phase-Flip Gate

Leaves `|0>` and `|1>` alone, but flips the sign of `|1>`. A π rotation around the Z-axis.

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

Key insight: Z does nothing *visible* to `|0>` or `|1>` alone — its effect only shows up once a
qubit is in superposition.


In [ ]:
Z = np.array([[1, 0],
              [0, -1]])

qc = QuantumCircuit(1)
qc.h(0)      # first reach |+>
qc.z(0)      # apply Z to qubit 0
display(qc.draw(output="mpl"))

sv = Statevector(qc)
print("Z|+> =", sv)   # should be |-> = (|0> - |1>)/sqrt(2)
plot_bloch_multivector(sv)

🎯 **Try It — A Gate You Can't See... Yet**

`Z|0> = |0>` and `Z|1> = -|1>`. A global minus sign on a basis state is unobservable — measuring
`|1>` or `-|1>` gives "1" with probability 1 either way.


In [ ]:
qc0 = QuantumCircuit(1)
qc0.z(0)
print("Z|0> =", Statevector(qc0))

qc1 = QuantumCircuit(1)
qc1.x(0)   # get to |1> first
qc1.z(0)
sv1 = Statevector(qc1)
print("Z|1> =", sv1)

# The state vectors -|1> and |1> differ only by an overall (global) phase.
# Compare their measurement probabilities -- identical, even though the vectors differ:
print("\nP(0), P(1) for Z|1>:", sv1.probabilities())

### Y — Bit-Flip + Phase-Flip

Combines what X and Z do. A π rotation around the Y-axis (out of the X–Z plane).

$$Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$$

`Y|0> = i|1>`, `Y|1> = -i|0>` — the bit flips exactly like X, and a factor of ±i appears, a phase,
like Z. Up to that phase, Y ≈ X then Z.


In [ ]:
Y = np.array([[0, -1j],
              [1j, 0]])

qc = QuantumCircuit(1)
qc.y(0)              # apply Y to qubit 0
display(qc.draw(output="mpl"))

sv = Statevector(qc)
print("Y|0> =", sv)
plot_bloch_multivector(sv)

# One relation to remember: Y = i * X * Z  (apply Z then X, then correct an overall phase i)
print("\nY == i * X @ Z ?", np.allclose(Y, 1j * (X @ Z)))

### Section Recap — Three Axes, Three Gates

- **X** — rotate π around X — bit-flip
- **Y** — rotate π around Y — bit + phase flip
- **Z** — rotate π around Z — phase-flip

All three are their own inverse.


In [ ]:
for name, M in [("X", X), ("Y", Y), ("Z", Z)]:
    print(f"{name}^2 = I ?", np.allclose(M @ M, np.eye(2)))

## Part 3 — The Hadamard Gate & Superposition

*The gate that makes quantum computing quantum.*

A qubit in superposition is described by a single vector with two nonzero amplitudes at once — not
two separate realities:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1$$

This is different from
classical uncertainty (a coin that's secretly heads or tails, we just don't know which): the
amplitudes are real and physical, and they can **interfere** — add up or cancel out — the way waves
do. That interference is the resource quantum algorithms exploit.


In [ ]:
# Normalization check: |alpha|^2 + |beta|^2 must equal 1 for any valid state
alpha, beta = 0.6, 0.8
print("|alpha|^2 + |beta|^2 =", alpha**2 + beta**2)   # a valid state
sv = Statevector([alpha, beta])
print(sv)

### H — The Superposition Maker

Rotates `|0>` to the equator — an equal superposition.

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

`H|0> = |+>` — H is a π rotation around the axis exactly halfway between X and Z, which is why it
turns a pole into an equator point.


In [ ]:
H = (1/np.sqrt(2)) * np.array([[1, 1],
                                [1, -1]])

qc = QuantumCircuit(1)
qc.h(0)              # apply H to qubit 0
display(qc.draw(output="mpl"))

sv = Statevector(qc)
plot_bloch_multivector(sv)

🎯 **Try It — Building |+>**


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
print(Statevector(qc))
# same magnitude on both amplitudes -> equal superposition

🤔 **Predict Before You Peek — What About H|1>?**

We just found `H|0> = |+>` (the equator). Where do you think H sends `|1>` — the same point, or
somewhere else on the equator? Think geometrically: H is a rotation around a fixed diagonal axis.


In [ ]:
qc = QuantumCircuit(1)
qc.x(0)   # get to |1> first
qc.h(0)   # then apply H
sv = Statevector(qc)
print("H|1> =", sv)
# H|1> = |-> = (|0> - |1>)/sqrt(2) -- same equator point family, opposite relative phase
plot_bloch_multivector(sv)

📊 **Let's Actually Measure It**

We'll run `H|0>` 1,000 times. Given `P(0) = P(1) = 1/2`, what split do you expect?

💡 **Note vs. the slide:** the slide shows `sampler.run(qc, shots=1000)`. `StatevectorSampler.run`
(a V2 primitive) expects an *iterable of circuits* ("pubs"), so the runnable version wraps the
circuit in a list — `sampler.run([qc], shots=1000)`, as below. Worth updating the slide's code
snippet to match if you want it to be copy-paste runnable for the audience.


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.measure_all()

sampler = StatevectorSampler()
job = sampler.run([qc], shots=1000)   # note the [qc] -- run() takes a list of circuits
counts = job.result()[0].data.meas.get_counts()

print(counts)
plot_histogram(counts)
# Run it yourself -- the exact numbers will differ each time you execute this cell.
# This is the experimental signature of superposition: a single qubit, prepared
# identically every time, genuinely produces both outcomes across many runs.

### What Happens If We Apply H Twice?


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)
print(Statevector(qc))
# back to |0> -- H^2 = I

### Superposition Scales Up Fast

Apply H to every qubit in an n-qubit register, and you get an equal superposition of all 2ⁿ
possible bitstrings — from a single line of code.


In [ ]:
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
display(qc.draw(output="mpl"))

sv = Statevector(qc)
print(sv)
# equal superposition of all 8 states: |000> ... |111>

probs = sv.probabilities()
print("\nAll 8 outcomes equally likely:", np.allclose(probs, probs[0]))
print("Each probability:", probs[0], " (1/8 =", 1/8, ")")

## Part 4 — The Phase Family: S and T

*Fine-grained phase turns — the fractional gears behind Z.*

Z, S, and T are all the same gate, just dialed to a different angle:

$$P(\varphi) = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\varphi} \end{pmatrix}, \qquad
P(\pi) = Z, \quad P(\pi/2) = S, \quad P(\pi/4) = T$$

`P(φ)` leaves `|0>` untouched, and multiplies the amplitude of `|1>` by `e^{iφ}` — a pure rotation
of its phase by angle φ.


In [ ]:
# Qiskit's qc.p(phi, 0) IS the general phase gate P(phi) -- confirm the three special cases
for phi, label in [(np.pi, "Z"), (np.pi/2, "S"), (np.pi/4, "T")]:
    p_qc = QuantumCircuit(1)
    p_qc.p(phi, 0)

    ref_qc = QuantumCircuit(1)
    getattr(ref_qc, label.lower())(0)

    match = np.allclose(Operator(p_qc).data, Operator(ref_qc).data)
    print(f"P({label} angle) == {label} gate ?  {match}")

### S — Quarter Turn (√Z)

Rotates the `|1>` phase by 90°. Apply it twice and you get Z exactly.

$$S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}, \qquad S \cdot S = Z$$


In [ ]:
S = np.array([[1, 0],
              [0, 1j]])

qc = QuantumCircuit(1)
qc.h(0)      # reach |+> first
qc.s(0)      # apply S to qubit 0
display(qc.draw(output="mpl"))

print("S|+> =", Statevector(qc))   # (|0> + i|1>) / sqrt(2)

print("\nS . S == Z ?", np.allclose(S @ S, Z))

### T — Eighth Turn (√S)

Rotates the `|1>` phase by 45° — nicknamed the "π/8 gate." The finest phase step we'll use.

$$T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}, \qquad T \cdot T = S, \quad T^4 = Z$$


In [ ]:
T = np.array([[1, 0],
              [0, np.exp(1j * np.pi / 4)]])

qc = QuantumCircuit(1)
qc.h(0)      # reach |+> first
qc.t(0)      # apply T to qubit 0
display(qc.draw(output="mpl"))

print("T|+> =", Statevector(qc))   # (|0> + e^{i pi/4}|1>) / sqrt(2)

print("\nT . T == S ?", np.allclose(T @ T, S))
print("T^4 == Z ?", np.allclose(T @ T @ T @ T, Z))

💡 **Why bother with such small angles?** `{H, S, T}` is a *universal gate set* — combined in
long enough sequences, just these three gates can approximate any possible single-qubit rotation to
arbitrary precision (the Solovay–Kitaev theorem guarantees this can be done efficiently). Real
quantum processors only implement a small native gate set, and every circuit you write gets
compiled ("transpiled") down into gates like these before it runs on hardware.


## Part 5 — Putting the Gates to Work

*From single gates to real circuits.*


### Order Matters — Gate Composition

Chaining gates in a circuit is matrix multiplication — and matrix multiplication does not commute.


In [ ]:
# Circuit A: X, then H
qcA = QuantumCircuit(1)
qcA.x(0)
qcA.h(0)

# Circuit B: H, then X
qcB = QuantumCircuit(1)
qcB.h(0)
qcB.x(0)

display(qcA.draw(output="mpl"))
display(qcB.draw(output="mpl"))

print("Combined matrix for Circuit A (H . X):\n", Operator(qcA).data)
print("\nCombined matrix for Circuit B (X . H):\n", Operator(qcB).data)
print("\nAre they the same matrix?", np.allclose(Operator(qcA).data, Operator(qcB).data))

### Closing the Loop — Remember Our Open Question?

Back in Part 2 we asked: what does X do to `|+>`? Let's finally find out, using order of
operations.


In [ ]:
# Circuit A: X then H, starting from |0>
qcA = QuantumCircuit(1)
qcA.x(0)
qcA.h(0)
print("|0> -X-> -H->", Statevector(qcA))   # should land on |->

# Circuit B: H then X, starting from |0>
qcB = QuantumCircuit(1)
qcB.h(0)
qcB.x(0)
print("|0> -H-> -X->", Statevector(qcB))   # should land back on |+>  -- X leaves |+> unchanged!

# |+> is an eigenstate of X: X|+> = (+1)|+>
print("\nX|+> == |+> ?", np.allclose(Statevector(qcB).data, Statevector.from_label('+').data))

# Bonus: what does Z do to |+> and |->?
qc_plus = QuantumCircuit(1); qc_plus.h(0); qc_plus.z(0)
print("\nZ|+> =", Statevector(qc_plus))

qc_minus = QuantumCircuit(1); qc_minus.x(0); qc_minus.h(0); qc_minus.z(0)
print("Z|-> =", Statevector(qc_minus))

### Full Worked Example — Reading a Real Circuit End to End


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.t(0)
qc.h(0)
display(qc.draw(output="mpl"))

print(Statevector(qc))
# reading it step by step:
# |0> -H-> |+> -T-> (|0> + e^{i pi/4}|1>)/sqrt(2) -H-> the state printed above

🤔 **Your Turn — Predict the Final State**

H, then Z, then H, starting from `|0>`. Using everything from today — the matrices, the Bloch
pictures, or just intuition — what state do you land on? No spoilers here! Build the circuit below
*before* looking at the reveal cell underneath it.


In [ ]:
# Your turn: build the H, Z, H circuit and print its Statevector
# qc = QuantumCircuit(1)
# qc.h(0)
# qc.z(0)
# qc.h(0)
# print(Statevector(qc))

✅ **Reveal** — only run this after you've made your prediction!


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.z(0)
qc.h(0)
display(qc.draw(output="mpl"))
print(Statevector(qc))
# H, Z, H turns out to be exactly the same transformation as a single X gate!

## Keep This Handy — The Single-Qubit Gate Cheat Sheet

| Gate | Matrix | Qiskit | Bloch action | One-liner |
|---|---|---|---|---|
| **X** | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | `qc.x(0)` | π around X | bit-flip |
| **Y** | $\begin{pmatrix}0&-i\\i&0\end{pmatrix}$ | `qc.y(0)` | π around Y | bit + phase flip |
| **Z** | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | `qc.z(0)` | π around Z | phase-flip |
| **H** | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | `qc.h(0)` | π, diagonal axis | makes superposition |
| **S** | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | `qc.s(0)` | π/2 around Z | quarter phase turn |
| **T** | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | `qc.t(0)` | π/4 around Z | eighth phase turn |
